# 02 - Isolating Variation

In [ ]:
%autosave 0

In [ ]:
import os
import sys
from pathlib import Path

# Set the base project directory
base_proj_dir = Path("/projects/site/pred/ihb-g-deco/USERS/adaml9/intestine_fate/notebooks/tf_ko_panel_analysis")

# Append necessary paths for module imports
sys.path.append("/projects/site/pred/ihb-g-deco/USERS/adaml9/bioinfo-blueprint/src/python")
sys.path.append(str(base_proj_dir))

import joblib
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
import rapids_singlecell as rsc
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import marsilea as ma
import marsilea.plotter as mp
import plotting.settings
from dynaconf import Dynaconf

# Load settings
settings = Dynaconf(
    settings_files=[base_proj_dir / 'settings.toml', base_proj_dir / '.secrets.toml'],
)

sc.settings.figdir = settings.ANALYSIS.figures_dir

In [ ]:
data_dir = Path(settings.IO.combined_data) / "all-sample" / "DGE_filtered"
anndata_dir = Path(settings.IO.anndata)

## Load data 

In [ ]:
adata = sc.read("/projects/site/pred/ihb-intestine-evo/lukas_area/eec_fate_project/tf_ko_screen/panel/models/contrastiveVI/adata_with_latent.h5ad")

In [ ]:
adata

## Perform dimensionality reduction

In [ ]:
rsc.pp.neighbors(adata, use_rep="salient_rep")
rsc.tl.umap(adata, key_added="X_umap_salient_rep")

In [ ]:
rsc.pp.neighbors(adata, use_rep="shared_rep")
rsc.tl.umap(adata, key_added="X_umap_shared_rep")

In [ ]:
for resolution in [0.3, 0.5, 1.0, 1.5, 2.0]:
    rsc.tl.leiden(adata, resolution=resolution, key_added=f"contrastivevi_shared_leiden_{resolution}")  

In [ ]:
sc.pl.embedding(adata, basis="X_umap_shared_rep", color="contrastivevi_shared_leiden_2.0", wspace=0.4, frameon=False, legend_loc="on data", show=True, save="_tfatlas_contrastivevi_shared_rep_leiden_2.0.pdf")

## Perform subclustering on the secretory branch

In [ ]:
import anndata as ad

def subcluster(
    adata: ad.AnnData,
    cluster_key: str,
    clusters_to_sub: list,
    new_key: str = "subcluster",
    use_rep: str = "shared_rep",
    resolution: float = 1.0,
):
    """
    Subcluster only selected clusters in an AnnData object.
    Keeps all other cluster labels unchanged.
    
    Parameters
    ----------
    adata : AnnData
        Input object with clustering in `.obs[cluster_key]`.
    cluster_key : str
        Column in .obs that stores the original clusters.
    clusters_to_sub : list
        Which cluster labels to subcluster (must match dtype in cluster_key).
    new_key : str, optional
        Name of new obs column to store results.
    use_rep : str, optional
        Which representation to use for neighbor graph construction.
    resolution : float, optional
        Resolution for Leiden clustering.
    """
    # start from original cluster labels
    adata.obs[new_key] = adata.obs[cluster_key].astype(str).copy()

    for c in clusters_to_sub:
        # subset AnnData to just one cluster
        subset = adata[adata.obs[cluster_key] == c].copy()

        # recompute neighbors + clustering
        rsc.pp.neighbors(subset, use_rep=use_rep)
        rsc.tl.leiden(subset, resolution=resolution, key_added="tmp_sub")

        # relabel: prefix with cluster name
        mapping = {
            cell: f"{c}_{sub}" for cell, sub in zip(subset.obs_names, subset.obs["tmp_sub"])
        }
        adata.obs.loc[subset.obs_names, new_key] = [
            mapping[cell] for cell in subset.obs_names
        ]

    return adata

In [ ]:
adata

In [ ]:
adata = subcluster(adata, cluster_key="contrastivevi_shared_leiden_2.0", clusters_to_sub=["21", "12", "23", "25", "19", "26", "27"], new_key="clust", resolution=0.3)

In [ ]:
sc.pl.embedding(adata, basis="X_umap_shared_rep", color="clust", wspace=0.4, frameon=False, legend_loc="on data", show=True)

In [ ]:
adata.write(anndata_dir / "tf_ko_panel_contrastiveVI.h5ad")

In [ ]:
adata = sc.read(anndata_dir / "tf_ko_panel_contrastiveVI.h5ad")

In [ ]:
adata = adata.raw.to_adata()

In [ ]:
adata.write(anndata_dir / "tf_ko_panel_contrastiveVI_raw.h5ad")

## Plot feature expression

In [ ]:
adata = sc.read(anndata_dir / "tf_ko_panel_contrastiveVI_raw.h5ad")

In [ ]:
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

In [ ]:
features = [
    "LGR5", "SMOC2", "OLFM4", "TLN2", "SI", "FABP2", "APOA4", 
    "MKI67", "TOP2A", "BEST4", "NOTCH2", "FCGBP", "MUC2", 
    "DEFA5", "DEFA6", "AVIL", "KIT", "NEUROG3", "PROX1", 
    "CHGA", "TPH1", "GHRL", "MLN", "SST", "HHEX", 
    "GIP", "RFX6", "NTS","CCK", "GCG", "PYY"
]
for feature in features:
    sc.pl.embedding(adata, basis="X_umap_shared_rep", color=feature, frameon=False, wspace=0.4, cmap="Purples", save=f"_tfatlas_contrastivevi_shared_rep_{feature}_expression.pdf", show=False, use_raw=False)

In [ ]:
features = [
    "LGR5", "SMOC2", "OLFM4", "TLN2", "SI", "FABP2", "APOA4", 
    "MKI67", "TOP2A", "BEST4", "NOTCH2", "FCGBP", "MUC2", 
    "DEFA5", "DEFA6", "AVIL", "KIT", "NEUROG3", "PROX1", 
    "CHGA", "TPH1", "GHRL", "MLN", "SST", "HHEX", 
    "GIP", "RFX6", "NTS","CCK", "GCG", "PYY"
]
for feature in features:
    sc.pl.embedding(adata[adata.obs["condition"].isin(["Control"])], basis="X_umap_shared_rep", color=feature, frameon=False, wspace=1, cmap="Purples", save=f"_tfatlas_contrastivevi_shared_rep_controls_{feature}_expression.pdf", show=False, use_raw=False)

In [ ]:
sc.pl.embedding(adata, basis="X_umap_shared_rep", color=["YAP1", "AXIN2", "RNF43", "EPHB2", "SOX9", "MYC", "SMARCA4", "MUC6"], wspace=0.4, frameon=False, legend_loc="on data", show=True, use_raw=False)

In [ ]:
sc.pl.embedding(adata, basis="X_umap_salient_rep", color="condition", wspace=0.4, frameon=False, legend_loc="on data", show=True, save="_tfatlas_contrastivevi_salient_rep_condition.pdf")